Cek apakah Nvidia CUDA tersedia

In [15]:
!nvidia-smi

Fri May  1 14:17:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             27W /   70W |     381MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Mounting google drive dengan notebook (Colab only)

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import yaml

YAML_FOLDER = '/content/drive/MyDrive/Training Data Colab/'
YAML_FILENAME = YAML_FOLDER + 'kicau_mania.yml'

with open(YAML_FILENAME, "r") as f:
    yaml_data = yaml.load(f, Loader=yaml.FullLoader)

model_name = yaml_data["name"]
model_path = yaml_data["path"]
training_data = yaml_data["training_data"]
hidden_layers = yaml_data["hidden_layer"]
labels = yaml_data["labels"]

print(model_name)
print(model_path)
print(training_data)
print(hidden_layers)
print(labels)

kicau_mania
kicau_mania.keras
kicau_mania.trdata.csv
[{'relu': 64}, {'dropout': 0.2}, {'relu': 64}, {'dropout': 0.2}, {'relu': 64}]
['KICAUUU', 'kicau_luar', 'Hi', 'Peace', 'F-you', 'Metal']


In [18]:
import pandas as pd

data = pd.read_csv(YAML_FOLDER + training_data, header=None)

X = data.iloc[:, 1:].values
Y = data.iloc[:, 0].values

In [19]:
print(X)
print(Y)

print(X.shape)
print(Y.shape)

[[-2.36959501e-01  1.00000000e+00  2.18737555e-08 ...  4.72690761e-02
   1.99481666e-01  4.36341319e-09]
 [-2.66189111e-01  1.00000000e+00  5.34877583e-07 ...  5.04278839e-02
   1.89443827e-01  1.01329256e-07]
 [-2.65516701e-01  1.00000000e+00  5.21863150e-07 ...  5.05847037e-02
   1.90514207e-01  9.94223441e-08]
 ...
 [ 8.95383947e-01  5.49651201e-01 -1.72425334e-07 ...  2.99314082e-01
   1.83740556e-01  5.76393298e-08]
 [ 8.97057538e-01  5.53509197e-01 -1.09925305e-07 ...  2.99107492e-01
   1.84557557e-01  3.66525903e-08]
 [ 8.93212755e-01  5.51259593e-01 -1.40082292e-07 ...  2.98489630e-01
   1.84217334e-01  4.68120405e-08]]
[0 0 0 ... 5 5 5]
(6000, 84)
(6000,)


In [20]:
import tensorflow as tf
from tensorflow.keras import models, layers

model = models.Sequential()
for layer_spec in hidden_layers:
    layer_type, params = next(iter(layer_spec.items()))
    if layer_type == "dropout":
        model.add(layers.Dropout(params))
    else:
        kwargs = {"activation": layer_type}
        if not model.layers:
            kwargs["input_shape"] = (X.shape[1],)
        model.add(layers.Dense(params, **kwargs))

model.add(layers.Dense(len(labels), activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [22]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │         5,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,150 (55.27 KB)

 Trainable params: 14,150 (55.27 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size= 0.2,
    shuffle=True,
    stratify=Y
)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [25]:
model.fit(X_train, Y_train, epochs=100, validation_data=(X_test, Y_test), callbacks=[es])

Epoch 1/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.6002 - loss: 1.0778 - val_accuracy: 0.9667 - val_loss: 0.3375
Epoch 2/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9073 - loss: 0.3033 - val_accuracy: 0.9825 - val_loss: 0.0949
Epoch 3/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9558 - loss: 0.1483 - val_accuracy: 0.9900 - val_loss: 0.0538
Epoch 4/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9725 - loss: 0.1028 - val_accuracy: 0.9900 - val_loss: 0.0431
Epoch 5/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9792 - loss: 0.0786 - val_accuracy: 0.9925 - val_loss: 0.0385
Epoch 6/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9798 - loss: 0.0710 - val_accuracy: 0.9908 - val_loss: 0.0293
Epoch 7/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9835 - loss: 0.0613 - val_accuracy: 0.9925 - val_loss: 0.0265
Epoch 8/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9869 - loss: 0.0534 - val_acc

In [26]:
model.save(model_path)